In [ ]:
import torch
!nvidia-smi
print("PyTorch:", torch.__version__)
print("CUDA kullanılabilir:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok")
assert torch.cuda.is_available(), "GPU görünmüyor; çalışma zamanı ayarını kontrol et."

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!git clone https://github.com/EfeAgaoglu/clothing-shelf-ai.git
%cd /content/clothing-shelf-ai
!git rev-parse HEAD

In [ ]:
from pathlib import Path
import hashlib, shutil, zipfile

source = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer/clothing-shelf-ai-0d7c381.zip")
local_zip = Path("/content/clothing-shelf-ai-0d7c381.zip")
project = Path("/content/clothing-shelf-ai")
expected = "d33fd4e63d211329900089b707ba73500a03ac5d16c56fb0e9a498311683aa1d"

assert source.is_file(), f"Drive'da ZIP bulunamadı: {source}"
assert not project.exists() or not any(project.iterdir()), f"Hedef klasör boş değil: {project}"

shutil.copy2(source, local_zip)
with local_zip.open("rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == expected, "Proje ZIP hash uyuşmuyor"

project.mkdir(exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    assert z.testzip() is None, "ZIP bozuk"
    z.extractall(project)

model_path = project / "runs/shelf_7class/finetune_20260910_143218/weights/best.pt"
with model_path.open("rb") as f:
    assert hashlib.file_digest(f, "sha256").hexdigest() == (
        "b08db72eb1dd5518c9113515518a850a8cfa6dbab36e48ce79fe24a7f418274d"
    ), "Checkpoint hash uyuşmuyor"

print("Proje ve son checkpoint hazır:", project)

In [ ]:
!python -m pip install -r requirements-portable.txt
!python -m pip check

In [ ]:
from pathlib import Path

project = Path("/content/clothing-shelf-ai")
requirements = project / "requirements-portable.txt"

print("Şu anki klasör:", Path.cwd())
print("Proje klasörü var:", project.is_dir())
print("Requirements var:", requirements.is_file())

In [ ]:
!python -m pip install -r /content/clothing-shelf-ai/requirements-portable.txt
!python -m pip check

In [ ]:
from pathlib import Path
import sys, torch
from ultralytics import YOLO

project = Path("/content/clothing-shelf-ai")
checkpoint = project / "runs/shelf_7class/finetune_20260910_143218/weights/best.pt"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA kullanılabilir:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok")
print("Checkpoint var:", checkpoint.is_file())

assert torch.cuda.is_available(), "Colab çalışma zamanında GPU seçili değil"
assert checkpoint.is_file(), "Proje ZIP'i veya checkpoint eksik"

model = YOLO(str(checkpoint))
print("Görev:", model.task)
print("Sınıflar:", model.names)

In [ ]:
!python -m pip install jedi
!python -m pip check

In [ ]:
from pathlib import Path
import hashlib, shutil, zipfile

project = Path("/content/clothing-shelf-ai")
transfer = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
source = transfer / "fashionpedia_balanced_10k_windows.zip"
sha_file = transfer / "fashionpedia_balanced_10k_windows.zip.sha256"
local_zip = Path("/content/fashionpedia_balanced_10k_windows.zip")
target = project / "data/fashionpedia_balanced_10k"
expected = "b2b4179c1a135aefa21e20efc4defe16b2adf810551d8d1a45796c1b0ff44add"

assert source.is_file() and sha_file.is_file(), "Drive'da dataset ZIP veya SHA dosyası eksik"
if not local_zip.exists():
    shutil.copy2(source, local_zip)

with local_zip.open("rb") as f:
    actual = hashlib.file_digest(f, "sha256").hexdigest()
assert actual == expected == sha_file.read_text().split()[0].lower(), "ZIP hash uyuşmuyor"
print("ZIP SHA-256 doğru")

if target.exists():
    print("Dataset klasörü zaten var; üzerine çıkarılmadı:", target)
else:
    with zipfile.ZipFile(local_zip) as z:
        assert all(n.startswith("data/fashionpedia_balanced_10k/") for n in z.namelist())
        z.extractall(project)
    print("Dataset çıkarıldı:", target)

In [ ]:
!python /content/clothing-shelf-ai/scripts/check_shelf_dataset.py \
  --data /content/clothing-shelf-ai/fashionpedia_balanced_10k.yaml

In [ ]:
!python /content/clothing-shelf-ai/scripts/train_shelf.py \
  --data /content/clothing-shelf-ai/fashionpedia_balanced_10k.yaml \
  --model /content/clothing-shelf-ai/runs/shelf_7class/finetune_20260910_143218/weights/best.pt \
  --device 0 \
  --batch 1 \
  --imgsz 512 \
  --epochs 1 \
  --dry-run

In [ ]:
from pathlib import Path
from datetime import datetime
import time, torch
from ultralytics import YOLO

project = Path("/content/clothing-shelf-ai")
checkpoint = project / "runs/shelf_7class/finetune_20260910_143218/weights/best.pt"
assert torch.cuda.is_available() and checkpoint.is_file()

torch.cuda.reset_peak_memory_stats(0)
started = time.perf_counter()

YOLO(str(checkpoint)).train(
    data=str(project / "fashionpedia_balanced_10k.yaml"),
    device=0, epochs=1, imgsz=512, batch=1,
    fraction=[200, 50],
    workers=0, optimizer="AdamW", lr0=0.0001, lrf=0.1,
    warmup_epochs=1.0, warmup_bias_lr=0.0,
    patience=10, seed=42, resume=False, cache=False,
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
    degrees=5.0, translate=0.05, scale=0.2,
    fliplr=0.5, flipud=0.0, hsv_s=0.3, hsv_v=0.2,
    val=True, save=True, plots=False, exist_ok=False,
    project=str(project / "runs/shelf_7class"),
    name="benchmark_t4_" + datetime.now().strftime("%Y%m%d_%H%M%S"),
)

print("BENCHMARK_SANIYE:", round(time.perf_counter() - started, 1))
print("TEPE_GPU_BELLEK_GB:", round(torch.cuda.max_memory_allocated(0) / 2**30, 2))

In [ ]:
from pathlib import Path
import yaml
from ultralytics.data.utils import check_det_dataset

project = Path("/content/clothing-shelf-ai")
dataset_root = project / "data/fashionpedia_balanced_10k"
assert (dataset_root / "images/train").is_dir()
assert (dataset_root / "images/val").is_dir()

config = yaml.safe_load((project / "fashionpedia_balanced_10k.yaml").read_text())
config["path"] = str(dataset_root.resolve())

colab_yaml = project / "runs/shelf_7class/colab_fashionpedia_balanced_10k.yaml"
colab_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

checked = check_det_dataset(str(colab_yaml), autodownload=False)
print("TRAIN:", checked["train"])
print("VAL:", checked["val"])

In [ ]:
from pathlib import Path
from datetime import datetime
import time
import torch
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

project = Path("/content/clothing-shelf-ai")
checkpoint = project / "runs/shelf_7class/finetune_20260910_143218/weights/best.pt"
data_yaml = project / "runs/shelf_7class/colab_fashionpedia_balanced_10k.yaml"

assert torch.cuda.is_available(), "Colab GPU görünmüyor"
assert checkpoint.is_file(), f"Checkpoint bulunamadı: {checkpoint}"
assert data_yaml.is_file(), f"Düzeltilmiş dataset YAML bulunamadı: {data_yaml}"

dataset = check_det_dataset(str(data_yaml), autodownload=False)
print("GPU:", torch.cuda.get_device_name(0))
print("Train yolu:", dataset["train"])
print("Validation yolu:", dataset["val"])
print("Benchmark: 200 train + 50 validation görüntüsü")

torch.cuda.reset_peak_memory_stats(0)
started = time.perf_counter()

YOLO(str(checkpoint)).train(
    data=str(data_yaml),
    device=0,
    epochs=1,
    imgsz=512,
    batch=1,
    fraction=[200, 50],
    workers=0,
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.1,
    warmup_epochs=1.0,
    warmup_bias_lr=0.0,
    patience=10,
    seed=42,
    resume=False,
    cache=False,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=5.0,
    translate=0.05,
    scale=0.2,
    fliplr=0.5,
    flipud=0.0,
    hsv_s=0.3,
    hsv_v=0.2,
    val=True,
    save=True,
    plots=False,
    exist_ok=False,
    project=str(project / "runs/shelf_7class"),
    name="benchmark_t4_" + datetime.now().strftime("%Y%m%d_%H%M%S"),
)

print("BENCHMARK_SANIYE:", round(time.perf_counter() - started, 1))
print("TEPE_GPU_BELLEK_GB:", round(torch.cuda.max_memory_allocated(0) / 2**30, 2))

In [ ]:
!python /content/clothing-shelf-ai/scripts/train_shelf.py \
  --data /content/clothing-shelf-ai/runs/shelf_7class/colab_fashionpedia_balanced_10k.yaml \
  --model /content/clothing-shelf-ai/runs/shelf_7class/finetune_20260910_143218/weights/best.pt \
  --device 0 --batch 1 --imgsz 512 --epochs 1 --execute

In [ ]:
from pathlib import Path
import hashlib, shutil, yaml

project = Path("/content/clothing-shelf-ai")
run = project / "runs/shelf_7class/finetune_20260923_093619"
backup = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer/trained_runs") / run.name

assert (run / "weights/best.pt").is_file()
assert (run / "weights/last.pt").is_file()
assert (run / "results.csv").is_file()
assert (run / "args.yaml").is_file()

backup.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run, backup, dirs_exist_ok=True)

args = yaml.safe_load((run / "args.yaml").read_text())
dataset_config = Path(args["data"])
if dataset_config.is_file():
    shutil.copy2(dataset_config, backup / "dataset.yaml")

for relative in ("weights/best.pt", "weights/last.pt", "results.csv", "args.yaml"):
    with (run / relative).open("rb") as f:
        source_hash = hashlib.file_digest(f, "sha256").hexdigest()
    with (backup / relative).open("rb") as f:
        backup_hash = hashlib.file_digest(f, "sha256").hexdigest()
    assert source_hash == backup_hash, f"Yedek doğrulanamadı: {relative}"
    print(relative, "OK", backup_hash[:16])

print("Drive yedeği hazır:", backup)

In [ ]:
from pathlib import Path
import hashlib, shutil

project = Path("/content/clothing-shelf-ai")
run = project / "runs/shelf_7class/finetune_20260923_093619"
backup = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer/trained_runs") / run.name

assert (run / "weights/best.pt").is_file(), "Yeni checkpoint Colab'da bulunamadı"
if not (backup / "weights/best.pt").is_file():
    backup.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(run, backup, dirs_exist_ok=True)

def sha256(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

assert sha256(run / "weights/best.pt") == sha256(backup / "weights/best.pt")
print("Drive yedeği doğrulandı:", backup)
print("Yeni best.pt SHA-256:", sha256(backup / "weights/best.pt"))

In [ ]:
from datetime import datetime
from pathlib import Path
import cv2
from ultralytics import YOLO

project = Path("/content/clothing-shelf-ai")
image_path = project / "test_images/shelf_test.jpg"

models = {
    "onceki": project / "runs/shelf_7class/finetune_20260910_143218/weights/best.pt",
    "yeni": project / "runs/shelf_7class/finetune_20260923_093619/weights/best.pt",
}

output_dir = (
    Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
    / ("shelf_comparison_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
)
output_dir.mkdir(parents=True)

for label, checkpoint in models.items():
    assert checkpoint.is_file(), f"Model bulunamadı: {checkpoint}"
    result = YOLO(str(checkpoint)).predict(
        source=str(image_path),
        device=0,
        conf=0.05,
        imgsz=1024,
        iou=0.50,
        max_det=300,
        save=False,
    )[0]

    output = output_dir / f"{label}_shelf_test.jpg"
    assert cv2.imwrite(str(output), result.plot())

    print(f"\n{label}: {len(result.boxes)} tespit")
    for cls_id, confidence in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
        print(f"  {result.names[int(cls_id)]}: %{confidence * 100:.2f}")
    print("Maske sayısı:", result.masks.data.shape[0] if result.masks is not None else 0)
    print("Maskeli görsel:", output)

print("\nKarşılaştırma klasörü:", output_dir)

In [ ]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile
import importlib.util
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

import torch
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

# Önceki Colab oturumu kapanmışsa proje arşivinden yeniden açılır.
drive_root = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
repo = Path("/content/clothing-shelf-ai")
archive = drive_root / "clothing-shelf-ai-0d7c381.zip"

if not repo.exists():
    assert archive.is_file(), f"Proje arşivi bulunamadı: {archive}"
    repo.mkdir(parents=True)
    with ZipFile(archive) as z:
        z.extractall(repo)

if importlib.util.find_spec("ultralytics") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "ultralytics==8.4.144"],
        check=True,
    )

from ultralytics import YOLO
import ultralytics

assert torch.cuda.is_available(), (
    "CUDA GPU bulunamadı. Colab'da Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seç."
)
print("GPU:", torch.cuda.get_device_name(0))
print("Ultralytics:", ultralytics.__version__)

models = {
    "deepfashion2": repo / "runs/segment/runs/deepfashion2_7class/test_3epoch-2/weights/best.pt",
    "fashionpedia_onceki": repo / "runs/shelf_7class/finetune_20260910_143218/weights/best.pt",
    "fashionpedia_yeni": drive_root / "trained_runs/finetune_20260923_093619/weights/best.pt",
}
for name, path in models.items():
    assert path.is_file(), f"{name} checkpoint bulunamadı: {path}"
    print(name, "→", path)

source_dir = drive_root / "shelf_eval_images"
sources = sorted(
    p for p in source_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".avif"}
)
assert sources, f"Test fotoğrafı bulunamadı: {source_dir}"
print("Test fotoğrafları:", [p.name for p in sources])

# AVIF dahil tüm girdileri Colab'ın geçici alanında JPEG'e dönüştür.
jpg_dir = Path("/content/shelf_eval_jpg")
jpg_dir.mkdir(parents=True, exist_ok=True)

def open_image(path):
    try:
        return Image.open(path)
    except Exception:
        if path.suffix.lower() != ".avif":
            raise
        # Pillow AVIF'i açamıyorsa yalnızca görüntü çözücü eklentisini kur.
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "pillow-avif-plugin==1.6.0"],
            check=True,
        )
        import pillow_avif
        return Image.open(path)

jpg_files = []
for source in sources:
    target = jpg_dir / f"{source.stem}.jpg"
    with open_image(source) as image:
        ImageOps.exif_transpose(image).convert("RGB").save(target, quality=95)
    jpg_files.append(target)
    print("Hazır:", source.name, "→", target.name)

output_dir = drive_root / (
    "shelf_eval_comparison_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
output_dir.mkdir(parents=True)

loaded_models = {}
expected_names = {
    0: "top", 1: "outwear", 2: "sleeveless_top",
    3: "shorts", 4: "trousers", 5: "skirt", 6: "dress"
}
for name, path in models.items():
    model = YOLO(str(path))
    assert model.task == "segment", f"{name}: segmentasyon modeli değil"
    assert model.names == expected_names, f"{name}: sınıf sırası farklı: {model.names}"
    loaded_models[name] = model

report = []
for image_path in jpg_files:
    panels = []
    report.append(f"\nFOTOĞRAF: {image_path.name}")

    for model_name, model in loaded_models.items():
        result = model.predict(
            source=str(image_path),
            device=0,
            conf=0.05,
            imgsz=1024,
            iou=0.50,
            max_det=300,
            save=False,
            verbose=False,
        )[0]

        plotted = Image.fromarray(result.plot()[..., ::-1]).convert("RGB")
        plotted.save(output_dir / f"{image_path.stem}__{model_name}.jpg", quality=92)
        plotted.thumbnail((550, 750))
        panels.append((model_name, plotted.copy()))

        detections = [
            f"{result.names[int(cls)]} %{float(conf) * 100:.1f}"
            for cls, conf in zip(result.boxes.cls, result.boxes.conf)
        ]
        mask_count = 0 if result.masks is None else len(result.masks.data)
        line = f"{model_name}: {len(detections)} tespit, {mask_count} maske; " + (
            ", ".join(detections) if detections else "tespit yok"
        )
        report.append(line)
        print(image_path.name, "—", line)

    panel_width = 570
    canvas = Image.new(
        "RGB",
        (panel_width * len(panels), max(im.height for _, im in panels) + 45),
        "white",
    )
    draw = ImageDraw.Draw(canvas)
    for index, (label, panel) in enumerate(panels):
        x = index * panel_width
        draw.text((x + 10, 10), label, fill="black")
        canvas.paste(panel, (x + 10, 35))

    comparison_path = output_dir / f"{image_path.stem}__KARSILASTIRMA.jpg"
    canvas.save(comparison_path, quality=92)
    print("Karşılaştırma:", comparison_path)
    display(canvas)

(output_dir / "tespit_ozeti.txt").write_text("\n".join(report), encoding="utf-8")
print("\nTÜM ÇIKTILAR:", output_dir)
print("Eğitim başlatılmadı.")

In [ ]:
from pathlib import Path
from datetime import datetime
from math import ceil
from collections import Counter
import importlib.util
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

import torch
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

if importlib.util.find_spec("ultralytics") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "ultralytics==8.4.144"],
        check=True,
    )

from ultralytics import YOLO

assert torch.cuda.is_available(), (
    "GPU bulunamadı. Colab'da Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seç."
)

root = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
model_path = root / "trained_runs/finetune_20260923_093619/weights/best.pt"
image_dir = root / "shelf_eval_images"

assert model_path.is_file(), f"Yeni checkpoint bulunamadı: {model_path}"
assert image_dir.is_dir(), f"Fotoğraf klasörü bulunamadı: {image_dir}"

images = sorted(
    p for p in image_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".avif"}
)
assert images, f"Fotoğraf bulunamadı: {image_dir}"

model = YOLO(str(model_path))
assert model.task == "segment"
assert model.names == {
    0: "top", 1: "outwear", 2: "sleeveless_top",
    3: "shorts", 4: "trousers", 5: "skirt", 6: "dress"
}

output_dir = root / ("shelf_tiling_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
output_dir.mkdir()
print("GPU:", torch.cuda.get_device_name(0))
print("Çıktı klasörü:", output_dir)

def open_rgb(path):
    try:
        with Image.open(path) as im:
            return ImageOps.exif_transpose(im).convert("RGB")
    except Exception:
        if path.suffix.lower() != ".avif":
            raise
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "pillow-avif-plugin==1.6.0"],
            check=True,
        )
        import pillow_avif
        with Image.open(path) as im:
            return ImageOps.exif_transpose(im).convert("RGB")

def predict(pil_image):
    return model.predict(
        source=pil_image,
        device=0,
        conf=0.05,
        imgsz=1024,
        iou=0.50,
        max_det=300,
        save=False,
        verbose=False,
    )[0]

def plotted_image(result):
    return Image.fromarray(result.plot()[..., ::-1]).convert("RGB")

report = []
for path in images:
    original = open_rgb(path)
    width, height = original.size

    # 1) Tam görüntü: önceki deneyle aynı tahmin ayarları.
    full_result = predict(original)
    full_plot = plotted_image(full_result)
    full_plot.save(output_dir / f"{path.stem}__tam.jpg", quality=92)

    full_items = [
        (full_result.names[int(cls)], float(conf))
        for cls, conf in zip(full_result.boxes.cls, full_result.boxes.conf)
    ]

    # 2) Yaklaşık 1200 piksel genişliğinde parçalar.
    # Küçük görüntülerde de 2x2 yakınlaştırma denenir.
    columns = max(2, ceil(width / 1200))
    rows = max(2, ceil(height / 1200))
    x_edges = [round(i * width / columns) for i in range(columns + 1)]
    y_edges = [round(i * height / rows) for i in range(rows + 1)]

    tiled_plot = Image.new("RGB", original.size)
    tile_items = []

    for row in range(rows):
        for col in range(columns):
            box = (
                x_edges[col], y_edges[row],
                x_edges[col + 1], y_edges[row + 1],
            )
            crop = original.crop(box)
            result = predict(crop)
            tiled_plot.paste(plotted_image(result), box[:2])

            tile_items.extend(
                (result.names[int(cls)], float(conf))
                for cls, conf in zip(result.boxes.cls, result.boxes.conf)
            )

    tiled_path = output_dir / f"{path.stem}__parcali.jpg"
    tiled_plot.save(tiled_path, quality=92)

    # Drive'da hızlı bakmak için yan yana küçük önizleme.
    preview_width = 650
    preview_height = round(height * preview_width / width)
    left = full_plot.resize((preview_width, preview_height))
    right = tiled_plot.resize((preview_width, preview_height))
    comparison = Image.new(
        "RGB", (preview_width * 2, preview_height + 35), "white"
    )
    comparison.paste(left, (0, 35))
    comparison.paste(right, (preview_width, 35))
    draw = ImageDraw.Draw(comparison)
    draw.text((10, 10), "TAM GORUNTU", fill="black")
    draw.text((preview_width + 10, 10), "PARCALI TAHMIN", fill="black")

    comparison_path = output_dir / f"{path.stem}__tam_vs_parcali.jpg"
    comparison.save(comparison_path, quality=92)

    full_counts = dict(Counter(label for label, _ in full_items))
    tile_counts = dict(Counter(label for label, _ in tile_items))
    lines = [
        f"\n{path.name} — {width}x{height}, {columns}x{rows} parça",
        f"Tam görüntü: {len(full_items)} tespit, sınıflar: {full_counts}",
        f"Parçalı: {len(tile_items)} tespit, sınıflar: {tile_counts}",
        f"Karşılaştırma: {comparison_path}",
    ]
    report.extend(lines)
    print("\n".join(lines))
    display(comparison)

(output_dir / "ozet.txt").write_text("\n".join(report), encoding="utf-8")
print("\nTÜM ÇIKTILAR:", output_dir)
print("Eğitim başlatılmadı.")

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
repo = Path("/content/clothing-shelf-ai")
project_zip = drive_root / "clothing-shelf-ai-0d7c381.zip"
data_zip = drive_root / "deepfashion2_7class_10k2k.zip"
sha_file = drive_root / "deepfashion2_7class_10k2k.zip.sha256"

assert data_zip.is_file(), f"ZIP bulunamadı: {data_zip}"
assert sha_file.is_file(), f"SHA dosyası bulunamadı: {sha_file}"

# Colab oturumu sıfırlandıysa yalnızca proje kodunu yeniden aç.
if not repo.exists():
    assert project_zip.is_file(), f"Proje arşivi bulunamadı: {project_zip}"
    repo.mkdir(parents=True)
    with ZipFile(project_zip) as z:
        z.extractall(repo)

assert (repo / "scripts/check_shelf_dataset.py").is_file(), (
    f"Proje dosyaları bulunamadı: {repo}"
)

expected = sha_file.read_text(encoding="utf-8").split()[0].lower()
digest = hashlib.sha256()
with data_zip.open("rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(block)
actual = digest.hexdigest()

print("Beklenen SHA-256:", expected)
print("Hesaplanan SHA-256:", actual)
assert actual == expected, "ZIP hash uyuşmuyor; arşivi açmadım."
print("SHA-256 doğru.")

with ZipFile(data_zip) as z:
    names = z.namelist()
    prefix = "data/deepfashion2_yolo_7class/"
    assert len(names) == 24000, f"ZIP'te {len(names)} giriş var; 24000 bekleniyordu."
    assert all(
        name.startswith(prefix)
        and ".." not in Path(name).parts
        and not name.startswith("/")
        for name in names
    ), "ZIP içinde beklenmeyen yol var; arşivi açmadım."
    assert z.testzip() is None, "ZIP CRC kontrolü başarısız; arşivi açmadım."
    print("ZIP bütünlüğü ve 24.000 dosya doğrulandı.")

    target = repo / "data/deepfashion2_yolo_7class"
    if target.exists():
        print("Dataset klasörü zaten var; üzerine yazmıyorum:", target)
    else:
        z.extractall(repo)
        print("Dataset açıldı:", target)

print("\nTam dataset kontrolü başlıyor; 12.000 görüntü için biraz sürebilir...")
subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/check_shelf_dataset.py"),
        "--data",
        str(repo / "deepfashion2_7class.yaml"),
    ],
    cwd=repo,
    check=True,
)

print("\nDeepFashion2 aktarımı doğrulandı. Eğitim başlatılmadı.")

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

import torch
import yaml

repo = Path("/content/clothing-shelf-ai")
drive_root = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")

df2 = repo / "data/deepfashion2_yolo_7class"
fp = repo / "data/fashionpedia_balanced_10k"
mixed = repo / "data/mixed_df2_fashionpedia_20k"
config = mixed / "dataset.yaml"
marker = mixed / "_ready.json"
checkpoint = drive_root / "trained_runs/finetune_20260923_093619/weights/best.pt"

assert (repo / "scripts/train_shelf.py").is_file(), (
    "Proje /content altında yok. Önce önceki aktarım hücresini yeniden çalıştır."
)
assert df2.is_dir(), (
    "DeepFashion2 verisi yok. Önce ZIP doğrulama/açma hücresini yeniden çalıştır."
)
assert checkpoint.is_file(), f"Yeni checkpoint bulunamadı: {checkpoint}"

# Colab oturumu sıfırlandıysa Fashionpedia ZIP'ini doğrulayıp yeniden aç.
if not fp.is_dir():
    fp_zip = drive_root / "fashionpedia_balanced_10k_windows.zip"
    fp_sha = drive_root / "fashionpedia_balanced_10k_windows.zip.sha256"
    assert fp_zip.is_file() and fp_sha.is_file(), "Fashionpedia ZIP veya SHA dosyası yok."

    expected_sha = fp_sha.read_text(encoding="utf-8").split()[0].lower()
    digest = hashlib.sha256()
    with fp_zip.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == expected_sha, "Fashionpedia ZIP hash uyuşmuyor."

    with ZipFile(fp_zip) as z:
        assert z.testzip() is None, "Fashionpedia ZIP bütünlük hatası."
        z.extractall(repo)
    print("Fashionpedia ZIP doğrulandı ve açıldı.")

sources = {
    "df2": (df2, {"train": 10000, "val": 2000}),
    "fp": (fp, {"train": 10000, "val": 1143}),
}

# Dört kaynak bölümündeki görüntü/etiket eşleşmesini önce kontrol et.
prepared = {}
for source_name, (source_root, counts) in sources.items():
    for split, expected_count in counts.items():
        image_dir = source_root / "images" / split
        label_dir = source_root / "labels" / split
        assert image_dir.is_dir() and label_dir.is_dir(), (
            f"Eksik kaynak klasör: {source_root}, {split}"
        )
        images = sorted(
            p for p in image_dir.iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
        )
        labels = sorted(label_dir.glob("*.txt"))
        assert len(images) == len(labels) == expected_count, (
            f"{source_name}/{split}: {len(images)} görüntü, "
            f"{len(labels)} etiket; beklenen {expected_count}"
        )
        assert {p.stem for p in images} == {p.stem for p in labels}, (
            f"{source_name}/{split}: görüntü ve etiket isimleri eşleşmiyor"
        )
        prepared[(source_name, split)] = (images, label_dir)
        print(f"{source_name}/{split}: {len(images)} eşleşen görüntü ve etiket")

# Tekrar çalıştırılırsa hazır kümeyi silmez veya yeniden yazmaz.
if mixed.exists():
    assert marker.is_file() and config.is_file(), (
        f"{mixed} var ama hazırlık tamamlanmamış. Burada durdum; klasörü silme, hatayı bana gönder."
    )
    print("Birleşik küme zaten hazır; üzerine yazmıyorum.")
else:
    for split in ("train", "val"):
        (mixed / "images" / split).mkdir(parents=True, exist_ok=False)
        (mixed / "labels" / split).mkdir(parents=True, exist_ok=False)

    for source_name, _ in sources.items():
        for split in ("train", "val"):
            images, label_dir = prepared[(source_name, split)]
            for image in images:
                new_stem = f"{source_name}__{image.stem}"
                new_image = mixed / "images" / split / (new_stem + image.suffix.lower())
                new_label = mixed / "labels" / split / (new_stem + ".txt")

                # Aynı /content diski üzerinde görüntünün yeni dosya adı;
                # görüntü baytları kopyalanmaz veya değiştirilmez.
                os.link(image, new_image)
                shutil.copy2(label_dir / (image.stem + ".txt"), new_label)

    dataset_config = {
        "path": str(mixed),
        "train": "images/train",
        "val": "images/val",
        "names": {
            0: "top", 1: "outwear", 2: "sleeveless_top",
            3: "shorts", 4: "trousers", 5: "skirt", 6: "dress"
        },
    }
    config.write_text(
        yaml.safe_dump(dataset_config, sort_keys=False),
        encoding="utf-8",
    )
    marker.write_text(
        json.dumps({"train": 20000, "val": 3143, "sources": ["df2", "fp"]}),
        encoding="utf-8",
    )
    print("Yeni birleşik küme hazırlandı:", mixed)

if importlib.util.find_spec("ultralytics") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "ultralytics==8.4.144"],
        check=True,
    )

assert torch.cuda.is_available(), "T4 GPU görünmüyor; eğitim hazırlığı için T4 çalışma zamanını seç."
print("GPU:", torch.cuda.get_device_name(0))
print("\nBirleşik dataset ve checkpoint dry-run kontrolü başlıyor...")

subprocess.run(
    [
        sys.executable,
        str(repo / "scripts/train_shelf.py"),
        "--data", str(config),
        "--model", str(checkpoint),
        "--device", "0",
        "--batch", "1",
        "--imgsz", "512",
        "--epochs", "1",
        "--dry-run",
    ],
    cwd=repo,
    check=True,
)

print("\nBirleşik veri hazırlığı tamamlandı. --execute kullanılmadı; eğitim başlatılmadı.")

In [ ]:
from pathlib import Path
from datetime import datetime
import yaml
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

repo = Path("/content/clothing-shelf-ai")
drive_root = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
config = repo / "data/mixed_df2_fashionpedia_20k/dataset.yaml"
checkpoint = drive_root / "trained_runs/finetune_20260923_093619/weights/best.pt"
output_root = drive_root / "trained_runs"

assert config.is_file(), (
    "Birleşik dataset yok. Önce veri hazırlama hücresini yeniden çalıştır."
)
assert checkpoint.is_file(), f"Başlangıç checkpoint'i yok: {checkpoint}"
assert torch.cuda.is_available(), "T4 GPU görünmüyor; eğitimi başlatmadım."
assert ultralytics.__version__ == "8.4.144", (
    f"Ultralytics sürümü değişmiş: {ultralytics.__version__}. Eğitimi başlatmadım."
)

data = yaml.safe_load(config.read_text(encoding="utf-8"))
expected_names = {
    0: "top", 1: "outwear", 2: "sleeveless_top",
    3: "shorts", 4: "trousers", 5: "skirt", 6: "dress"
}
assert data["names"] == expected_names, "7 sınıf sırası yanlış; eğitimi başlatmadım."
assert Path(data["path"]).resolve() == config.parent.resolve(), (
    "Dataset yolu yanlış; eğitimi başlatmadım."
)

for split, expected in (("train", 20000), ("val", 3143)):
    folder = config.parent / "images" / split
    count = sum(p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
                for p in folder.iterdir())
    assert count == expected, (
        f"{split}: {count} görüntü var, {expected} bekleniyordu; eğitimi başlatmadım."
    )

check_det_dataset(str(config), autodownload=False)

model = YOLO(str(checkpoint))
assert model.task == "segment", "Checkpoint segmentasyon modeli değil."
assert model.names == expected_names, "Checkpoint sınıf sırası yanlış."

run_name = "mixed_df2_fp_2epoch_" + datetime.now().strftime("%Y%m%d_%H%M%S")
output_root.mkdir(parents=True, exist_ok=True)
run_dir = output_root / run_name
assert not run_dir.exists(), f"Çıktı klasörü zaten var: {run_dir}"

print("GPU:", torch.cuda.get_device_name(0))
print("Başlangıç modeli:", checkpoint)
print("Dataset:", config)
print("Train: 20.000 | Val: 3.143 | Epoch: 2")
print("Yeni çıktı klasörü:", run_dir)
print("Eğitim şimdi başlıyor...")

model.train(
    data=str(config),
    epochs=2,
    imgsz=512,
    batch=1,
    device=0,
    workers=0,
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.1,
    warmup_epochs=1.0,
    warmup_bias_lr=0.0,
    patience=10,
    seed=42,
    resume=False,
    cache=False,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=5.0,
    translate=0.05,
    scale=0.2,
    fliplr=0.5,
    flipud=0.0,
    hsv_s=0.3,
    hsv_v=0.2,
    plots=True,
    val=True,
    save=True,
    save_period=1,
    fraction=1.0,
    project=str(output_root),
    name=run_name,
    exist_ok=False,
)

print("\nEğitim tamamlandı.")
for filename in ("best.pt", "last.pt"):
    path = run_dir / "weights" / filename
    print(filename, "var:", path.is_file(), "yol:", path)
print("Sonuç klasörü:", run_dir)

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

import torch
from PIL import Image, ImageOps, ImageDraw
from ultralytics import YOLO
from IPython.display import display

root = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
repo = Path("/content/clothing-shelf-ai")
photos = root / "shelf_eval_images"

checkpoints = {
    "onceki_fashionpedia": root / "trained_runs/finetune_20260923_093619/weights/best.pt",
    "yeni_karma": root / "trained_runs/mixed_df2_fp_2epoch_20260923_110902/weights/best.pt",
}
assert torch.cuda.is_available(), "GPU görünmüyor."
assert photos.is_dir(), f"Fotoğraf klasörü yok: {photos}"
for name, path in checkpoints.items():
    assert path.is_file(), f"{name} checkpoint yok: {path}"

sources = sorted(
    p for p in photos.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".avif"}
)
shelf_test = repo / "test_images/shelf_test.jpg"
if shelf_test.is_file():
    sources.append(shelf_test)
assert sources, "Test fotoğrafı bulunamadı."

def open_rgb(path):
    try:
        with Image.open(path) as im:
            return ImageOps.exif_transpose(im).convert("RGB")
    except Exception:
        if path.suffix.lower() != ".avif":
            raise
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "pillow-avif-plugin==1.6.0"],
            check=True,
        )
        import pillow_avif
        with Image.open(path) as im:
            return ImageOps.exif_transpose(im).convert("RGB")

models = {name: YOLO(str(path)) for name, path in checkpoints.items()}
expected_names = {
    0: "top", 1: "outwear", 2: "sleeveless_top",
    3: "shorts", 4: "trousers", 5: "skirt", 6: "dress"
}
for name, model in models.items():
    assert model.task == "segment" and model.names == expected_names, (
        f"{name}: model görevi veya sınıf sırası yanlış"
    )

output = root / ("shelf_compare_mixed_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
output.mkdir()
summary = []

for source in sources:
    image = open_rgb(source)
    panels = []
    summary.append(f"\nFOTOĞRAF: {source.name}")

    for name, model in models.items():
        result = model.predict(
            source=image,
            device=0,
            conf=0.05,
            imgsz=1024,
            iou=0.50,
            max_det=300,
            save=False,
            verbose=False,
        )[0]

        plotted = Image.fromarray(result.plot()[..., ::-1]).convert("RGB")
        plotted.save(output / f"{source.stem}__{name}.jpg", quality=92)
        plotted.thumbnail((650, 850))
        panels.append((name, plotted.copy()))

        detections = [
            f"{result.names[int(cls)]} %{float(conf) * 100:.1f}"
            for cls, conf in zip(result.boxes.cls, result.boxes.conf)
        ]
        masks = 0 if result.masks is None else len(result.masks.data)
        line = f"{name}: {len(detections)} tespit, {masks} maske; " + (
            ", ".join(detections) if detections else "tespit yok"
        )
        summary.append(line)
        print(source.name, "—", line)

    width = 670
    canvas = Image.new(
        "RGB",
        (width * 2, max(panel.height for _, panel in panels) + 40),
        "white",
    )
    draw = ImageDraw.Draw(canvas)
    for index, (name, panel) in enumerate(panels):
        x = index * width
        draw.text((x + 10, 10), name, fill="black")
        canvas.paste(panel, (x + 10, 35))

    path = output / f"{source.stem}__KARSILASTIRMA.jpg"
    canvas.save(path, quality=92)
    display(canvas)

(output / "tespit_ozeti.txt").write_text("\n".join(summary), encoding="utf-8")
print("\nTÜM ÇIKTILAR:", output)
print("Yeni eğitim başlatılmadı.")

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "ultralytics==8.4.144",
    "pillow-avif-plugin",
])

import torch
import ultralytics
from ultralytics import YOLO
import pillow_avif  # AVIF raf fotoğrafları için

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

import torch
from PIL import Image, ImageOps, ImageDraw
from ultralytics import YOLO
from IPython.display import display

root = Path("/content/drive/MyDrive/clothing-shelf-ai-transfer")
repo = Path("/content/clothing-shelf-ai")
photos = root / "shelf_eval_images"

checkpoints = {
    "onceki_fashionpedia": root / "trained_runs/finetune_20260923_093619/weights/best.pt",
    "yeni_karma": root / "trained_runs/mixed_df2_fp_2epoch_20260923_110902/weights/best.pt",
}
assert torch.cuda.is_available(), "GPU görünmüyor."
assert photos.is_dir(), f"Fotoğraf klasörü yok: {photos}"
for name, path in checkpoints.items():
    assert path.is_file(), f"{name} checkpoint yok: {path}"

sources = sorted(
    p for p in photos.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".avif"}
)
shelf_test = repo / "test_images/shelf_test.jpg"
if shelf_test.is_file():
    sources.append(shelf_test)
assert sources, "Test fotoğrafı bulunamadı."

def open_rgb(path):
    try:
        with Image.open(path) as im:
            return ImageOps.exif_transpose(im).convert("RGB")
    except Exception:
        if path.suffix.lower() != ".avif":
            raise
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "pillow-avif-plugin==1.6.0"],
            check=True,
        )
        import pillow_avif
        with Image.open(path) as im:
            return ImageOps.exif_transpose(im).convert("RGB")

models = {name: YOLO(str(path)) for name, path in checkpoints.items()}
expected_names = {
    0: "top", 1: "outwear", 2: "sleeveless_top",
    3: "shorts", 4: "trousers", 5: "skirt", 6: "dress"
}
for name, model in models.items():
    assert model.task == "segment" and model.names == expected_names, (
        f"{name}: model görevi veya sınıf sırası yanlış"
    )

output = root / ("shelf_compare_mixed_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
output.mkdir()
summary = []

for source in sources:
    image = open_rgb(source)
    panels = []
    summary.append(f"\nFOTOĞRAF: {source.name}")

    for name, model in models.items():
        result = model.predict(
            source=image,
            device=0,
            conf=0.05,
            imgsz=1024,
            iou=0.50,
            max_det=300,
            save=False,
            verbose=False,
        )[0]

        plotted = Image.fromarray(result.plot()[..., ::-1]).convert("RGB")
        plotted.save(output / f"{source.stem}__{name}.jpg", quality=92)
        plotted.thumbnail((650, 850))
        panels.append((name, plotted.copy()))

        detections = [
            f"{result.names[int(cls)]} %{float(conf) * 100:.1f}"
            for cls, conf in zip(result.boxes.cls, result.boxes.conf)
        ]
        masks = 0 if result.masks is None else len(result.masks.data)
        line = f"{name}: {len(detections)} tespit, {masks} maske; " + (
            ", ".join(detections) if detections else "tespit yok"
        )
        summary.append(line)
        print(source.name, "—", line)

    width = 670
    canvas = Image.new(
        "RGB",
        (width * 2, max(panel.height for _, panel in panels) + 40),
        "white",
    )
    draw = ImageDraw.Draw(canvas)
    for index, (name, panel) in enumerate(panels):
        x = index * width
        draw.text((x + 10, 10), name, fill="black")
        canvas.paste(panel, (x + 10, 35))

    path = output / f"{source.stem}__KARSILASTIRMA.jpg"
    canvas.save(path, quality=92)
    display(canvas)

(output / "tespit_ozeti.txt").write_text("\n".join(summary), encoding="utf-8")
print("\nTÜM ÇIKTILAR:", output)
print("Yeni eğitim başlatılmadı.")

In [ ]:
!apt-get -qq update && apt-get -qq install -y gh

from google.colab import userdata
import os
os.environ["GH_TOKEN"] = userdata.get("GITHUB_TOKEN")

!gh auth status
!gh auth setup-git

In [ ]:
from google.colab import userdata
import os

os.environ["GH_TOKEN"] = userdata.get("GITHUB_TOKEN")

!gh auth status
!gh auth setup-git

In [ ]:
from pathlib import Path
import subprocess

project = Path("/content/clothing-shelf-ai")
assert project.is_dir(), f"Proje klasörü bulunamadı: {project}"

subprocess.run(["gh", "auth", "setup-git"], check=True)

check = subprocess.run(
    ["git", "-C", str(project), "rev-parse", "--show-toplevel"],
    capture_output=True, text=True
)

if check.returncode == 0 and Path(check.stdout.strip()).resolve() == project.resolve():
    print("Git kaydı bulundu.")
    for args in [
        ["status", "--short"],
        ["branch", "--show-current"],
        ["log", "-1", "--oneline"],
    ]:
        subprocess.run(["git", "-C", str(project), *args], check=True)
else:
    print("Bu proje klasöründe Git kaydı yok.")
    print("Mevcut dosyaları koruyarak ayrı bir GitHub klonu hazırlamamız gerekiyor.")

print("\nProje kökündeki notebook dosyaları:")
print([p.name for p in project.glob("*.ipynb")])

In [ ]:
from pathlib import Path
from google.colab import drive, userdata
import os
import subprocess

drive.mount("/content/drive")
os.environ["GH_TOKEN"] = userdata.get("GITHUB_TOKEN")

subprocess.run(["gh", "auth", "setup-git"], check=True)

project = Path("/content/clothing-shelf-ai")

if not project.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/EfeAgaoglu/clothing-shelf-ai.git",
        str(project)
    ], check=True)
elif not (project / ".git").exists():
    raise RuntimeError(
        "Klasör mevcut fakat Git kaydı yok. Dosyaların üzerine yazılmadı."
    )

print("\nPROJE:", project)
subprocess.run(
    ["git", "-C", str(project), "log", "-1", "--oneline"],
    check=True
)
subprocess.run(
    ["git", "-C", str(project), "status", "--short"],
    check=True
)

transfer = Path(
    "/content/drive/MyDrive/clothing-shelf-ai-transfer"
)

print("\nEĞİTİM KAYITLARI:")
runs = transfer / "trained_runs"
if runs.is_dir():
    for run in sorted(runs.iterdir()):
        if run.is_dir():
            print(run.name)
            for name in ("results.csv", "args.yaml", "weights/best.pt"):
                path = run / name
                print(f"  {name}: {'VAR' if path.is_file() else 'YOK'}")
else:
    print("trained_runs klasörü bulunamadı.")

print("\nNOTEBOOK DOSYALARI:")
found = []
for folder in [
    Path("/content/drive/MyDrive/Colab Notebooks"),
    transfer,
]:
    if folder.is_dir():
        found.extend(sorted(folder.glob("*.ipynb")))

for notebook in found:
    print(notebook)

if not found:
    print("Bu iki klasörün içinde notebook bulunamadı.")

print("\nKontrol tamamlandı. Eğitim veya push başlatılmadı.")